In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold
from scipy.sparse import hstack, csr_matrix
from gensim.models import Word2Vec
from nltk.stem import PorterStemmer
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train['text'] = train['title'].fillna('') + ' ' + train['body'].fillna('')
test['text'] = test['title'].fillna('') + ' ' + test['body'].fillna('')

print(f"Train: {len(train)} samples, Test: {len(test)} samples")
print(f"Positive class: {train['label'].sum()} ({train['label'].mean()*100:.1f}%)")

Train: 10800 samples, Test: 973 samples
Positive class: 2070.0 (19.2%)


Предобработка

In [4]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

train['clean'] = train['text'].apply(clean_text)
test['clean'] = test['text'].apply(clean_text)


TF-IDF

In [5]:
tfidf = TfidfVectorizer(max_features=7000, ngram_range=(1, 2), min_df=2, max_df=0.7, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train['clean'])
X_test_tfidf = tfidf.transform(test['clean'])

In [7]:
def get_simple_features(df):
    features = pd.DataFrame()
    features['len_words'] = df['clean'].str.split().str.len()
    features['has_suicide'] = df['clean'].str.contains('suicid|kill|die|death', na=False).astype(int)
    return features

train_feat = get_simple_features(train)
test_feat = get_simple_features(test)

Word2Vec

In [ ]:
train_tokens = [text.split() for text in train['clean']]
test_tokens = [text.split() for text in test['clean']]

all_tokens = train_tokens + test_tokens
w2v = Word2Vec(sentences=all_tokens, vector_size=150, window=7, min_count=2, workers=4, epochs=12, sg=1, seed=42)

def get_doc_vector(tokens, model, size=150):
    if not tokens:
        return np.zeros(size)
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(size)

train_w2v = np.array([get_doc_vector(t, w2v) for t in train_tokens])
test_w2v = np.array([get_doc_vector(t, w2v) for t in test_tokens])

In [9]:
X_train = hstack([
    X_train_tfidf,
    csr_matrix(train_feat.values),
    csr_matrix(train_w2v)
]).tocsr()

X_test = hstack([
    X_test_tfidf,
    csr_matrix(test_feat.values),
    csr_matrix(test_w2v)
]).tocsr()

y = train['label'].values
print(f"Размерность: {X_train.shape}")

Размерность: (10800, 7152)


Поиск оптимального порога

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_weights = (0.5, 0.25, 0.25)

all_probs = []
all_labels = []

for train_idx, val_idx in skf.split(X_train, y):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    
    lr_fold = LogisticRegression(C=2.5, class_weight='balanced', max_iter=2000, solver='liblinear', random_state=42)
    rf_fold = RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_split=3, class_weight='balanced', random_state=42)
    gb_fold = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42)
    
    lr_fold.fit(X_tr, y_tr)
    rf_fold.fit(X_tr, y_tr)
    gb_fold.fit(X_tr, y_tr)
    
    probs_lr = lr_fold.predict_proba(X_val)[:, 1]
    probs_rf = rf_fold.predict_proba(X_val)[:, 1]
    probs_gb = gb_fold.predict_proba(X_val)[:, 1]
    
    ensemble_probs = best_weights[0] * probs_lr + best_weights[1] * probs_rf + best_weights[2] * probs_gb
    all_probs.extend(ensemble_probs)
    all_labels.extend(y_val)

prec, rec, thresholds = precision_recall_curve(all_labels, all_probs)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores[:-1])] if len(thresholds) > 0 else 0.48

print(f"Оптимальный порог: {best_threshold:.3f}")
print(f"CV F1 при этом пороге: {max(f1_scores[:-1]):.4f}")


Оптимальный порог: 0.514
CV F1 при этом пороге: 0.8100


In [ ]:
lr_final = LogisticRegression(C=2.5, class_weight='balanced', max_iter=2000, solver='liblinear', random_state=42)
rf_final = RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_split=3, class_weight='balanced', random_state=42, n_jobs=-1)
gb_final = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42)

lr_final.fit(X_train, y)
rf_final.fit(X_train, y)
gb_final.fit(X_train, y)

preds_lr = lr_final.predict_proba(X_test)[:, 1]
preds_rf = rf_final.predict_proba(X_test)[:, 1]
preds_gb = gb_final.predict_proba(X_test)[:, 1]

final_probs = (best_weights[0] * preds_lr + best_weights[1] * preds_rf + best_weights[2] * preds_gb)

final_preds = (final_probs > best_threshold).astype(int)

In [14]:
submission = pd.DataFrame({'id': test['id'], 'label': final_preds})
submission.to_csv('submission.csv', index=False)

print("Модель: Voting Classifier (LR + RF + GB)")
print("Признаки: TF-IDF + Word2Vec + длина + suicide")
print(f"Веса: LR={best_weights[0]}, RF={best_weights[1]}, GB={best_weights[2]}")
print(f"Оптимальный порог: {best_threshold:.3f}")
print(f"Предсказания: 0={sum(final_preds==0)}, 1={sum(final_preds==1)}")
print(f"Процент positive: {sum(final_preds==1)/len(final_preds)*100:.1f}%")

Модель: Voting Classifier (LR + RF + GB)
Признаки: TF-IDF + Word2Vec + длина + suicide
Веса: LR=0.5, RF=0.25, GB=0.25
Оптимальный порог: 0.514
Предсказания: 0=734, 1=239
Процент positive: 24.6%
